# Flowers-102 MaskMix + VPT — experiments notebook

Runs on Colab T4 via the Cursor remote-kernel bridge. Cell 1 clones the repo and mounts Drive; subsequent cells drive `src.train.train_one_config` from YAML configs. Later cells (Tasks 12–15.5) will be appended as experiment blocks are rolled out.

In [ ]:
# %% Cell 1 — one-time per Colab session
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/huytruong2004/SC4001-FinalProject.git"
REPO_DIR = Path("/content/repo")

try:
    from google.colab import drive  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    # 1. Mount Drive for checkpoint / results persistence across session drops.
    drive.mount("/content/drive")
    PERSIST_ROOT = Path("/content/drive/MyDrive/sc4001_flowers102")
    PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

    # 2. Clone or update the repo on the Colab VM.
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

    # 3. pip install dependencies.
    subprocess.run(["pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

    # 4. Make src/ importable.
    sys.path.insert(0, str(REPO_DIR))

    REPO_ROOT = REPO_DIR
    # Paths that must persist across sessions go on Drive.
    CKPT_ROOT = PERSIST_ROOT / "checkpoints"
    # Data is ~550 MB: cache on Drive so we don't re-download every session.
    DATA_ROOT = PERSIST_ROOT / "data" / "flowers-102"
else:
    # Local-kernel fallback (for CPU debugging only).
    REPO_ROOT = Path().resolve()
    if REPO_ROOT.name == "notebooks":
        REPO_ROOT = REPO_ROOT.parent
    sys.path.insert(0, str(REPO_ROOT))
    CKPT_ROOT = REPO_ROOT / "checkpoints"
    DATA_ROOT = REPO_ROOT / "data" / "flowers-102"

# Results / figures live in the cloned repo so we can git push them back.
RESULTS_JSONL = REPO_ROOT / "results" / "runs.jsonl"
FIGURES_DIR = REPO_ROOT / "figures"
for p in (CKPT_ROOT, DATA_ROOT, RESULTS_JSONL.parent, FIGURES_DIR):
    p.mkdir(parents=True, exist_ok=True)

import torch
print(f"Repo : {REPO_ROOT}")
print(f"Data : {DATA_ROOT}")
print(f"Ckpts: {CKPT_ROOT}")
print(f"Torch: {torch.__version__}  CUDA: {torch.cuda.is_available()}  "
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'}")

In [ ]:
# %%
from scripts.download_masks import download
download(DATA_ROOT)

In [ ]:
# %%
import numpy as np
import matplotlib.pyplot as plt
from src.data import Flowers102WithMasks

ds = Flowers102WithMasks(root=DATA_ROOT, split="train", image_size=224, train_augment=False)
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, idx in zip(axes, rng.integers(0, len(ds), 5)):
    img, mask, label = ds[int(idx)]
    # Unnormalise
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_show = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img_show)
    ax.imshow(mask[0].numpy(), alpha=0.4, cmap="Reds")
    ax.set_title(f"class {label}")
    ax.axis("off")
plt.suptitle("Mask / image alignment sanity check")
plt.savefig(FIGURES_DIR / "sanity_mask_alignment.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# %%
from src.train import train_one_config

result_D = train_one_config(
    config_path=REPO_ROOT / "configs/D_linear_probe.yaml",
    seed=0,
    data_root=DATA_ROOT,
    checkpoint_dir=CKPT_ROOT,
    results_path=RESULTS_JSONL,
)
print(result_D)

In [ ]:
# %%
from src.train import train_one_config

BLOCK_A_CONFIGS = [
    "configs/A1_baseline.yaml",
    "configs/A2_maskmix.yaml",
    "configs/A3_attsup.yaml",
    "configs/A4_ours.yaml",
]
SEEDS = [0, 1, 2]

block_a_results = []
for cfg in BLOCK_A_CONFIGS:
    for seed in SEEDS:
        print(f"\n=== {cfg} seed={seed} ===")
        r = train_one_config(
            config_path=REPO_ROOT / cfg,
            seed=seed,
            data_root=DATA_ROOT,
            checkpoint_dir=CKPT_ROOT,
            results_path=RESULTS_JSONL,
        )
        block_a_results.append(r)
        print(r)

import pandas as pd
df_a = pd.DataFrame(block_a_results)
df_a.to_csv(REPO_ROOT / "results/block_a.csv", index=False)
df_a